In [ ]:
!pip install rank-bm25
import kagglehub
import pandas as pd
import re
import os
import spacy
import numpy as np
from rank_bm25 import BM25Okapi


In [ ]:
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Path to dataset files: /kaggle/input/ag-news-classification-dataset


In [ ]:
print("Files in dataset directory:")
for f in os.listdir(path):
    print(f)

Files in dataset directory:
.nfs000000004a6c997a00000035
train.csv
test.csv


In [ ]:
data = pd.read_csv("/kaggle/input/ag-news-classification-dataset/test.csv")
data.head(50)

,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...
5,4,Open Letter Against British Copyright Indoctri...,The British Department for Education and Skill...
6,4,Loosing the War on Terrorism,"\\""Sven Jaschan, self-confessed author of the ..."
7,4,"FOAFKey: FOAF, PGP, Key Distribution, and Bloo...",\\FOAF/LOAF and bloom filters have a lot of i...
8,4,E-mail scam targets police chief,"Wiltshire Police warns about ""phishing"" after ..."
9,4,"Card fraud unit nets 36,000 cards","In its first two years, the UK's dedicated car..."


In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = re.sub(r'#\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]

    return " ".join(tokens)


In [ ]:
data_sample = data.copy()
data_sample['clean_text'] = (data_sample['Title'].fillna('') + " " +
                             data_sample['Description'].fillna('')).apply(preprocess_text)

In [ ]:
tokenized_corpus = [doc.split() for doc in data_sample['clean_text']]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
queries = [
    "Methods for detecting cancer symptoms",
    "Next space mission after Columbia",
    "Apple launches new product",
    "Who are gold medal winners in Athens?",
    "UN mediating Middle East conflicts",
    "Urgent climate issues worldwide",
    "Airplane accident in Latin America",
    "New technology against spam",
    "dictator court case",
    "Terroristic act in Russia"

]
for i, query in  enumerate(queries, 1):
    print(f"\nResults for query {i}: {query}")
    print("=" * 80)

    query_tokens = preprocess_text(query).split()
    scores = bm25.get_scores(query_tokens)
    top_n = 5
    top_n_idx = np.argsort(scores)[::-1][:top_n]

    for idx in top_n_idx:
      print(f"Score: {scores[idx]:.2f}")
      print(f"Title: {data_sample['Title'].iloc[idx]}")
      print(f"Description: {data_sample['Description'].iloc[idx]}")
      print("-" * 80)


Results for query 1: Methods for detecting cancer symptoms
Score: 16.59
Title: Dogs in Training to Sniff Out Cancer
Description: Experts have trained unwanted dogs into supersniffers that can detect drugs or bombs. Now they're focusing on a new threat #151;prostate cancer.
--------------------------------------------------------------------------------
Score: 13.31
Title: Bud Selig Has Skin Cancer Surgery
Description: NEW YORK -- Baseball commissioner Bud Selig had surgery Monday to remove a cancerous lesion from his forehead. The lesion was detected last month during Selig #39;s annual physical, and a biopsy confirmed that it contained melanoma, a form of skin cancer.
--------------------------------------------------------------------------------
Score: 10.34
Title: 'Cure' a 4-Letter Word for Cancer Doctors
Description: At a time when more people are cured of cancer than ever before, fewer doctors seem willing to say so. They call the cancer undetectable, or in remission. They tell 

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
doc_embeddings = model.encode(
    data_sample['clean_text'].tolist(),
    convert_to_tensor=True
)

In [ ]:
query_embeddings = model.encode(
    queries,
    convert_to_tensor=True
)

In [ ]:
similarity_scores = util.cos_sim(query_embeddings, doc_embeddings)

In [ ]:
top_k = 5
for q_idx, query in enumerate(queries):

    scores = similarity_scores[q_idx]

    top_results = torch.topk(scores, k=top_k)

    print(f"\nQuery: {query}")
    print("=" * 60)

    for score, idx in zip(top_results.values, top_results.indices):

        idx = idx.item()
        score = score.item()

        print(f"Score: {score:.4f}")
        print("Title:", data_sample['Title'].iloc[idx])
        print("Description:", data_sample['Description'].iloc[idx])
        print("-" * 60)


Query: Methods for detecting cancer symptoms
Score: 0.3869
Title: Dogs said to smell cancer signs 
Description: LONDON -- It has long been suspected that man's best friend has a special ability to sense when something is wrong with us. Now, the first experiment to verify that scientifically has demonstrated that dogs are able to smell cancer.
------------------------------------------------------------
Score: 0.3533
Title: Older mobiles may cause tumours: study
Description: The Institute of Environmental Medicine (IMM) at Karolinska Institute in Sweden found no indications of risk for less than 10 years of usage.
------------------------------------------------------------
Score: 0.3422
Title: Dogs in Training to Sniff Out Cancer
Description: Experts have trained unwanted dogs into supersniffers that can detect drugs or bombs. Now they're focusing on a new threat #151;prostate cancer.
------------------------------------------------------------
Score: 0.3385
Title: Obesity Raises Risk

In [ ]:
!pip install faiss-cpu
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.6 MB/s eta 0:00:00


In [ ]:
doc_embeddings = model.encode(data_sample['clean_text'].tolist(), convert_to_numpy=True)

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
query_embedding = model.encode(queries, convert_to_numpy=True)
top_k = 5
distances, indices = index.search(query_embedding, top_k)

In [ ]:
for q_idx, query in enumerate(queries):
    print(f"Query: {query}")
    print("=" * 60)

    for rank, doc_idx in enumerate(indices[q_idx], start=1):
        print(f"Rank: {rank}")
        print(f"Distance: {distances[q_idx][rank-1]:.2f}")
        print("Title:", data_sample['Title'].iloc[doc_idx])
        print("Description:", data_sample['Description'].iloc[doc_idx])
        print("-" * 60)

    print("\n")


Query: Methods for detecting cancer symptoms
Rank: 1
Distance: 1.23
Title: Dogs said to smell cancer signs 
Description: LONDON -- It has long been suspected that man's best friend has a special ability to sense when something is wrong with us. Now, the first experiment to verify that scientifically has demonstrated that dogs are able to smell cancer.
------------------------------------------------------------
Rank: 2
Distance: 1.29
Title: Older mobiles may cause tumours: study
Description: The Institute of Environmental Medicine (IMM) at Karolinska Institute in Sweden found no indications of risk for less than 10 years of usage.
------------------------------------------------------------
Rank: 3
Distance: 1.32
Title: Dogs in Training to Sniff Out Cancer
Description: Experts have trained unwanted dogs into supersniffers that can detect drugs or bombs. Now they're focusing on a new threat #151;prostate cancer.
------------------------------------------------------------
Rank: 4
Distan